In [3]:
import os
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score

class GenderPredictor:
    def __init__(self, model_path, input_size=(100, 100)):
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        self.input_size = input_size
        
    def preprocess_image(self, image_path):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            return None
        image = cv2.resize(image, self.input_size)
        image = np.expand_dims(image, axis=-1).astype(np.float32) / 255.0
        return image
    
    def predict(self, image_path):
        image = self.preprocess_image(image_path)
        if image is None:
            return None
        
        input_data = np.expand_dims(image, axis=0).astype(
            self.input_details[0]['dtype']
        )
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        self.interpreter.invoke()
        output = self.interpreter.get_tensor(self.output_details[0]['index'])
        
        return np.argmax(output[0])
    
    def extract_label(self, filename):
        """Extract gender label from filename (man/woman)"""
        filename_lower = filename.lower()
        
        # Check woman BEFORE man (because "man" is in "woman")
        if 'woman' in filename_lower:
            return 1  # Female
        elif 'man' in filename_lower:
            return 0  # Male
        else:
            return None
    
    def process_folder(self, folder_path):
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        image_files = [f for f in os.listdir(folder_path) 
                      if Path(f).suffix.lower() in image_extensions]
        
        true_labels = []
        pred_labels = []
        
        for filename in image_files:
            image_path = os.path.join(folder_path, filename)
            
            # Extract true label from filename
            true_label = self.extract_label(filename)
            if true_label is None:
                print(f"✗ {filename}: Could not extract label")
                continue
            
            # Predict
            pred = self.predict(image_path)
            if pred is None:
                print(f"✗ {filename}: Could not read image")
                continue
            
            true_labels.append(true_label)
            pred_labels.append(pred)
            pred_name = 'Male' if pred == 0 else 'Female'
            true_name = 'Male' if true_label == 0 else 'Female'
            print(f"✓ {filename}: True={true_name}, Pred={pred_name}")
        
        return true_labels, pred_labels


# Main execution
if __name__ == "__main__":
    MODEL_PATH = "/kaggle/input/models/adityamodi20/model/tflite/default/1/gender-model.tflite"
    IMAGE_FOLDER = "/kaggle/input/models/adityamodi20/testing-image/tflite/default/1/hualcosa DLBAIPEAI01_edge_ai main evaluation_images"
    
    if not os.path.exists(MODEL_PATH):
        print(f"✗ Model not found: {MODEL_PATH}")
        exit(1)
    
    if not os.path.isdir(IMAGE_FOLDER):
        print(f"✗ Folder not found: {IMAGE_FOLDER}")
        exit(1)
    
    predictor = GenderPredictor(MODEL_PATH)
    true_labels, pred_labels = predictor.process_folder(IMAGE_FOLDER)
    
    if len(true_labels) == 0:
        print("✗ No valid images processed")
        exit(1)
    
    # Print metrics
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT ON EDGE DEVICE")
    print("="*60)
    print(f"\nAccuracy: {accuracy_score(true_labels, pred_labels):.4f}\n")
    print(classification_report(true_labels, pred_labels, 
                               target_names=['Male', 'Female']))


2026-03-25 12:13:49.934017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774440830.216739      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774440830.312777      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774440831.080534      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774440831.080573      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774440831.080578      55 computation_placer.cc:177] computation placer alr

✓ young_woman_sad_3.jpg: True=Female, Pred=Male
✓ young_woman_happy_2.jpg: True=Female, Pred=Female
✓ elderly_woman_happy_3.jpg: True=Female, Pred=Female
✓ young_man_happy_3.jpg: True=Male, Pred=Male
✓ young_man_sad_1.jpg: True=Male, Pred=Male
✓ elderly_man_happy_1.jpg: True=Male, Pred=Male
✓ young_man_sad_2.jpg: True=Male, Pred=Male
✓ young_woman_sad_1.jpg: True=Female, Pred=Female
✓ elderly_man_sad_3.jpg: True=Male, Pred=Male
✓ elderly_man_happy_2.jpg: True=Male, Pred=Male
✓ elderly_man_happy_3.jpg: True=Male, Pred=Male
✓ young_man_happy_1.jpg: True=Male, Pred=Male
✓ elderly_man_sad_2.jpg: True=Male, Pred=Male
✓ young_woman_happy_1.jpg: True=Female, Pred=Female
✓ young_man_happy_2.jpg: True=Male, Pred=Male
✓ young_woman_happy_3.jpg: True=Female, Pred=Female
✓ elderly_woman_sad_1.jpg: True=Female, Pred=Male
✓ elderly_woman_sad_3.jpg: True=Female, Pred=Male
✓ elderly_woman_happy_1.jpg: True=Female, Pred=Female
✓ young_man_sad_3.jpg: True=Male, Pred=Male
✓ elderly_woman_happy_2.jpg: Tru

In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score

class EmotionPredictor:
    def __init__(self, model_path, input_size=(200, 200), normalize_type='0-1'):
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        self.input_size = input_size
        self.normalize_type = normalize_type  # '0-1' or '-1-1' or 'uint8'
        self.emotion_labels = ['ahegao', 'angry', 'happy', 'neutral', 'sad', 'surprise']
        
        print(f"\n=== Model Input Details ===")
        print(f"Shape: {self.input_details[0]['shape']}")
        print(f"Data type: {self.input_details[0]['dtype']}")
        print(f"Quantization: {self.input_details[0]['quantization']}")
        print(f"Normalization type: {normalize_type}\n")
        
    def preprocess_image(self, image_path):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            return None
        
        image = cv2.resize(image, self.input_size)
        image = np.expand_dims(image, axis=-1).astype(np.float32)
        
        # Apply different normalization based on setting
        if self.normalize_type == '0-1':
            image = image / 255.0
        elif self.normalize_type == '-1-1':
            image = (image / 127.5) - 1.0
        elif self.normalize_type == 'uint8':
            image = image.astype(np.uint8)
        
        return image
    
    def predict(self, image_path):
        image = self.preprocess_image(image_path)
        if image is None:
            return None
        
        input_data = np.expand_dims(image, axis=0).astype(
            self.input_details[0]['dtype']
        )
        
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        self.interpreter.invoke()
        output = self.interpreter.get_tensor(self.output_details[0]['index'])
        
        return np.argmax(output[0])
    
    def extract_label(self, filename):
        """Extract emotion label from filename"""
        filename_lower = filename.lower()
        
        for emotion in self.emotion_labels:
            if emotion in filename_lower:
                return self.emotion_labels.index(emotion)
        
        return None
    
    def process_folder(self, folder_path):
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        image_files = [f for f in os.listdir(folder_path) 
                      if Path(f).suffix.lower() in image_extensions]
        
        true_labels = []
        pred_labels = []
        
        for filename in image_files:
            image_path = os.path.join(folder_path, filename)
            
            # Extract true label from filename
            true_label = self.extract_label(filename)
            if true_label is None:
                print(f"✗ {filename}: Could not extract emotion label")
                continue
            
            # Predict
            pred = self.predict(image_path)
            if pred is None:
                print(f"✗ {filename}: Could not read image")
                continue
            
            true_labels.append(true_label)
            pred_labels.append(pred)
            pred_emotion = self.emotion_labels[pred]
            true_emotion = self.emotion_labels[true_label]
            status = "✓" if pred == true_label else "✗"
            print(f"{status} {filename}: True={true_emotion}, Pred={pred_emotion}")
        
        return true_labels, pred_labels


# Main execution
if __name__ == "__main__":
    MODEL_PATH = "/kaggle/input/models/adityamodi20/model/tflite/default/1/emotion-model.tflite"
    IMAGE_FOLDER = "/kaggle/input/models/adityamodi20/testing-image/tflite/default/1/hualcosa DLBAIPEAI01_edge_ai main evaluation_images"
    
    if not os.path.exists(MODEL_PATH):
        print(f"✗ Model not found: {MODEL_PATH}")
        exit(1)
    
    if not os.path.isdir(IMAGE_FOLDER):
        print(f"✗ Folder not found: {IMAGE_FOLDER}")
        exit(1)
    
    # Try different normalization types
    normalize_types = ['0-1', '-1-1', 'uint8']
    
    for norm_type in normalize_types:
        print(f"\n{'='*60}")
        print(f"Testing with normalization: {norm_type}")
        print(f"{'='*60}")
        
        predictor = EmotionPredictor(MODEL_PATH, normalize_type=norm_type)
        true_labels, pred_labels = predictor.process_folder(IMAGE_FOLDER)
        
        if len(true_labels) == 0:
            print("✗ No valid images processed")
            continue
        
        # Print metrics
        print(f"\n{'='*60}")
        print(f"EMOTION CLASSIFICATION REPORT ON EDGE DEVICE ({norm_type})")
        print(f"{'='*60}")
        print(f"\nAccuracy: {accuracy_score(true_labels, pred_labels):.4f}\n")
        print(classification_report(true_labels, pred_labels, 
                                   target_names=predictor.emotion_labels,
                                   zero_division=0))


2026-03-26 13:12:19.481187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774530739.775209      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774530739.853226      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774530740.509336      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774530740.509444      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774530740.509448      55 computation_placer.cc:177] computation placer alr


Testing with normalization: 0-1

=== Model Input Details ===
Shape: [  1 200 200   1]
Data type: <class 'numpy.float32'>
Quantization: (0.0, 0)
Normalization type: 0-1



/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


✗ young_woman_sad_3.jpg: True=sad, Pred=happy
✓ young_woman_happy_2.jpg: True=happy, Pred=happy
✗ elderly_woman_happy_3.jpg: True=happy, Pred=sad
✗ young_man_happy_3.jpg: True=happy, Pred=ahegao
✗ young_man_sad_1.jpg: True=sad, Pred=ahegao
✗ elderly_man_happy_1.jpg: True=happy, Pred=sad
✗ young_man_sad_2.jpg: True=sad, Pred=happy
✗ young_woman_sad_1.jpg: True=sad, Pred=happy
✓ elderly_man_sad_3.jpg: True=sad, Pred=sad
✓ elderly_man_happy_2.jpg: True=happy, Pred=happy
✗ elderly_man_happy_3.jpg: True=happy, Pred=surprise
✗ young_man_happy_1.jpg: True=happy, Pred=neutral
✓ elderly_man_sad_2.jpg: True=sad, Pred=sad
✗ young_woman_happy_1.jpg: True=happy, Pred=surprise
✗ young_man_happy_2.jpg: True=happy, Pred=ahegao
✗ young_woman_happy_3.jpg: True=happy, Pred=angry
✗ elderly_woman_sad_1.jpg: True=sad, Pred=neutral
✗ elderly_woman_sad_3.jpg: True=sad, Pred=neutral
✗ elderly_woman_happy_1.jpg: True=happy, Pred=sad
✗ young_man_sad_3.jpg: True=sad, Pred=ahegao
✗ elderly_woman_happy_2.jpg: True=

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✗ elderly_man_happy_3.jpg: True=happy, Pred=neutral
✗ young_man_happy_1.jpg: True=happy, Pred=neutral
✓ elderly_man_sad_2.jpg: True=sad, Pred=sad
✗ young_woman_happy_1.jpg: True=happy, Pred=neutral
✗ young_man_happy_2.jpg: True=happy, Pred=angry
✗ young_woman_happy_3.jpg: True=happy, Pred=sad
✓ elderly_woman_sad_1.jpg: True=sad, Pred=sad
✓ elderly_woman_sad_3.jpg: True=sad, Pred=sad
✗ elderly_woman_happy_1.jpg: True=happy, Pred=angry
✓ young_man_sad_3.jpg: True=sad, Pred=sad
✗ elderly_woman_happy_2.jpg: True=happy, Pred=sad
✓ elderly_man_sad_1.jpg: True=sad, Pred=sad
✓ elderly_woman_sad_2.jpg: True=sad, Pred=sad
✓ young_woman_sad_2.jpg: True=sad, Pred=sad

EMOTION CLASSIFICATION REPORT ON EDGE DEVICE (-1-1)

Accuracy: 0.3750



ValueError: Number of classes, 4, does not match size of target_names, 6. Try specifying the labels parameter

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

label_mapping = {
    'child': 0, 'teen': 1, 'young': 2, 'middle': 3,
    'mature': 4, 'senior': 5, 'older': 6, 'elderly': 7,
}

reverse_mapping = {v: k for k, v in label_mapping.items()}

class AgePredictor:
    def __init__(self, model_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.model = torch.jit.load(model_path, map_location=device)
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    
    def predict(self, image_path):
        try:
            image = Image.open(image_path).convert('RGB')
            x = self.transform(image).unsqueeze(0).to(self.device)
            with torch.no_grad():
                output = self.model(x)
                pred_class = torch.argmax(output, dim=1).item()
            return pred_class
        except:
            return None

def extract_true_label(filename):
    filename_lower = filename.lower()
    for label in label_mapping.keys():
        if label in filename_lower:
            return label_mapping[label]
    return None

def test_model(model_path, image_folder):
    predictor = AgePredictor(model_path)
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    image_files = [f for f in os.listdir(image_folder) 
                   if os.path.splitext(f)[1].lower() in image_extensions]
    
    true_labels = []
    pred_labels = []
    
    for filename in image_files:
        true_label = extract_true_label(filename)
        if true_label is None:
            continue
        
        image_path = os.path.join(image_folder, filename)
        pred_label = predictor.predict(image_path)
        
        if pred_label is not None:
            true_labels.append(true_label)
            pred_labels.append(pred_label)
            print(f"{filename}: True={reverse_mapping[true_label]}, Pred={reverse_mapping[pred_label]}")
    
    # Get unique classes present in predictions
    unique_classes = sorted(set(true_labels + pred_labels))
    class_names = [reverse_mapping[c] for c in unique_classes]
    
    # Classification Report
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT ON EDGE DEVICE")
    print("="*60)
    print(classification_report(true_labels, pred_labels, 
                                target_names=class_names,
                                labels=unique_classes))
    

    # Overall Accuracy
    print(f"\nOverall Accuracy: {accuracy_score(true_labels, pred_labels):.4f}")

# Usage
if __name__ == "__main__":
    test_model("/kaggle/input/models/adityamodi20/age-cpu/pytorch/default/1/age_classifier-cpu.pt", "/kaggle/input/models/adityamodi20/testing-image/tflite/default/1/hualcosa DLBAIPEAI01_edge_ai main evaluation_images")


young_woman_sad_3.jpg: True=young, Pred=young
young_woman_happy_2.jpg: True=young, Pred=teen
elderly_woman_happy_3.jpg: True=elderly, Pred=elderly
young_man_happy_3.jpg: True=young, Pred=elderly
young_man_sad_1.jpg: True=young, Pred=teen
elderly_man_happy_1.jpg: True=elderly, Pred=elderly
young_man_sad_2.jpg: True=young, Pred=senior
young_woman_sad_1.jpg: True=young, Pred=teen
elderly_man_sad_3.jpg: True=elderly, Pred=elderly
elderly_man_happy_2.jpg: True=elderly, Pred=elderly
elderly_man_happy_3.jpg: True=elderly, Pred=elderly
young_man_happy_1.jpg: True=young, Pred=elderly
elderly_man_sad_2.jpg: True=elderly, Pred=elderly
young_woman_happy_1.jpg: True=young, Pred=elderly
young_man_happy_2.jpg: True=young, Pred=elderly
young_woman_happy_3.jpg: True=young, Pred=elderly
elderly_woman_sad_1.jpg: True=elderly, Pred=elderly
elderly_woman_sad_3.jpg: True=elderly, Pred=elderly
elderly_woman_happy_1.jpg: True=elderly, Pred=elderly
young_man_sad_3.jpg: True=young, Pred=elderly
elderly_woman_ha

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
